In [0]:
from pyspark.sql.functions import col

fact_df = spark.table("medical_project.gold.fact_encounters")
date_df = spark.table("medical_project.gold.dim_date")

In [0]:
from pyspark.sql.functions import to_date, quarter, when

df = fact_df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

# Add quarter
df = df.withColumn(
    "quarter",
    quarter(col("encounter_date"))
)

# Cast year and month to int
df = df.withColumn("year", col("year").cast("int")) \
       .withColumn("month", col("month").cast("int"))

# Add readable duration category for KPI 4
df = df.withColumn(
    "duration_category",
    when(col("is_over_24_hours") == 1, "More than 24 hours")
    .otherwise("24 hours or less")
)

In [0]:
from pyspark.sql.functions import count, sum, col, coalesce, lit

cube_for_save = df.cube(
    "year",
    "month",
    "quarter",
    "payer_id",
    "encounter_class",
    "duration_category"
).agg(
    count("encounter_id").alias("encounter_count"),
    sum("total_cost").alias("total_cost")
)

cube_for_save = cube_for_save \
    .withColumn("year", coalesce(col("year").cast("string"), lit("ALL"))) \
    .withColumn("month", coalesce(col("month").cast("string"), lit("ALL"))) \
    .withColumn("quarter", coalesce(col("quarter").cast("string"), lit("ALL"))) \
    .withColumn("payer_id", coalesce(col("payer_id"), lit("ALL"))) \
    .withColumn("encounter_class", coalesce(col("encounter_class"), lit("ALL"))) \
    .withColumn("duration_category", coalesce(col("duration_category"), lit("ALL")))

cube_for_save.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.encounter_cube")

In [0]:
display(cube_for_save)